# Scaffold analysis

This notebook derives Bemis-Murcko scaffolds from the curated modeling dataset, summarizes scaffold-level activity, and saves the scaffold summary used in the manuscript outputs.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

ROOT = Path("..").resolve()
DATA = ROOT / "data" / "processed" / "leishmania_ml_dataset.csv"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
df[["Molecule ChEMBL ID", "SMILES", "pIC50"]].head()


In [ ]:
def bemis_murcko_scaffold(smiles: str):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold is None or scaffold.GetNumAtoms() == 0:
        return None
    return Chem.MolToSmiles(scaffold, isomericSmiles=True)

scaffold_df = df.copy()
scaffold_df["Scaffold"] = scaffold_df["SMILES"].map(bemis_murcko_scaffold)
scaffold_df = scaffold_df.dropna(subset=["Scaffold"])
scaffold_df[["SMILES", "Scaffold", "pIC50"]].head()


In [ ]:
scaffold_summary = (
    scaffold_df.groupby("Scaffold")
    .agg(count=("pIC50", "size"), mean=("pIC50", "mean"), median=("pIC50", "median"))
    .query("count >= 3")
    .sort_values(["mean", "count"], ascending=[False, False])
    .reset_index()
)

scaffold_summary.to_csv(RESULTS / "scaffold_activity_summary.csv", index=False)
scaffold_summary.head(20)


In [ ]:
top_scaffolds = scaffold_summary.head(20).sort_values("mean")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(top_scaffolds)), top_scaffolds["mean"], color="#0F766E")
ax.set_yticks(range(len(top_scaffolds)))
ax.set_yticklabels(top_scaffolds["Scaffold"], fontsize=7)
ax.set_xlabel("Mean pIC50")
ax.set_title("Top recurrent scaffolds by mean anti-leishmanial activity")
fig.tight_layout()
fig.savefig(RESULTS / "top20_scaffolds.png", dpi=300, bbox_inches="tight")
plt.show()
